# ENM404 Üretim Çizelgeleme — Grup 9
## Flow-Shop Scheduling Probleminin Tabu Search ile Optimizasyonu

**İstanbul Kültür Üniversitesi | 2024–2025**

---

| | |
|---|---|
| **Grup** | 9 |
| **Problem** | Flow-Shop Scheduling |
| **Algoritma** | Tabu Search (TS) |
| **Benchmark** | Taillard (1993) |

---

### İçindekiler
1. Problem Tanımı
2. Makespan Hesaplama
3. Benchmark Instance Üretici (Taillard)
4. NEH Başlangıç Sezgiseli
5. Komşuluk Yapısı
6. Tabu Search Algoritması
7. Parametre Tuning
8. Resmi Deneyler (10 Run × 5 Instance)
9. Karşılaştırmalı Analiz
10. Sonuçlar

---
## 1. Problem Tanımı

**Flow-Shop Scheduling:** n iş ve m makineden oluşan bir sistemde tüm işlerin aynı makine sırasıyla işlenmesi problemidir.

**Amaç:** Makespan (Cmax) değerini minimize et

$$C_{max} = C(\pi(n), m)$$

**Kısıtlar:**
- Her makine aynı anda yalnızca bir iş işler
- Her iş aynı makine sırasını izler  
- Preemption (kesme) yoktur

---
## 2. Makespan Hesaplama

In [ ]:
def calculate_makespan(permutation, processing_times):
    """
    Flow-Shop Scheduling için Cmax (makespan) hesaplar.
    
    Args:
        permutation: İş sırası listesi, örn. [2, 0, 4, 1, 3]
        processing_times: p[i][j] matrisi (n_machines x n_jobs)
    Returns:
        Cmax (int)
    """
    n_machines = len(processing_times)
    n_jobs = len(permutation)

    C = [0] * n_jobs
    prev = [0] * n_jobs

    for i in range(n_machines):
        row = processing_times[i]
        c_ik_prev = 0
        for k in range(n_jobs):
            job = permutation[k]
            c_ik_prev = max(c_ik_prev, prev[k]) + row[job]
            C[k] = c_ik_prev
        prev = C[:]

    return C[n_jobs - 1]


# ── Test ──────────────────────────────────────────────────────────────────────
pt_test = [[3, 2, 5], [4, 1, 3]]
perm_test = [0, 1, 2]
cmax = calculate_makespan(perm_test, pt_test)
print(f"Test makespan: {cmax}  (Beklenen: 15)")

---
## 3. Taillard Benchmark Instance Üretici

Taillard (1993) makalesindeki orijinal LCG (Linear Congruential Generator) kullanılarak
benchmark instance'ları üretilmektedir. Bu sayede literatürdeki BKS değerleriyle karşılaştırma yapılabilir.

In [ ]:
import os

def taillard_rng(seed):
    """Taillard'ın orijinal rastgele sayı üreticisi (LCG)."""
    seed = (seed * 536870923 + 1) % (2**31 - 1)
    return seed, seed / (2**31 - 1)


def generate_taillard_instance(n_jobs, n_machines, seed_init, p_min=1, p_max=99):
    """Taillard'ın orijinal yöntemiyle işlem süresi matrisi üretir."""
    seed = seed_init
    processing_times = []
    for i in range(n_machines):
        row = []
        for j in range(n_jobs):
            seed, r = taillard_rng(seed)
            p = int(r * (p_max - p_min + 1)) + p_min
            row.append(p)
        processing_times.append(row)
    return processing_times


# ── Taillard Benchmark Kataloğu ───────────────────────────────────────────────
TAILLARD_CATALOG = {
    'tai20x5':  {'n_jobs': 20,  'n_machines': 5,  'seed': 873654221, 'bks': 1278},
    'tai50x10': {'n_jobs': 50,  'n_machines': 10, 'seed': 379008056, 'bks': 2724},
    'tai100x10':{'n_jobs': 100, 'n_machines': 10, 'seed': 878272350, 'bks': 5770},
    'tai100x20':{'n_jobs': 100, 'n_machines': 20, 'seed': 715671442, 'bks': 6286},
    'tai200x20':{'n_jobs': 200, 'n_machines': 20, 'seed': 760290334, 'bks': 11294},
}

BKS = {name: info['bks'] for name, info in TAILLARD_CATALOG.items()}

# Instance'ları üret ve kaydet
os.makedirs('instances', exist_ok=True)
INSTANCES = {}

for name, info in TAILLARD_CATALOG.items():
    pt = generate_taillard_instance(info['n_jobs'], info['n_machines'], info['seed'])
    INSTANCES[name] = pt
    print(f"✓ {name:12s} | {info['n_jobs']}x{info['n_machines']} | BKS={info['bks']}")

print(f"\nToplam {len(INSTANCES)} instance üretildi.")

---
## 4. NEH Başlangıç Sezgiseli

Nawaz-Enscore-Ham (1983) sezgiseli, Flow-Shop için literatürde en etkili başlangıç yöntemi olarak kabul görmektedir.

**Adımlar:**
1. İşleri toplam işlem sürelerine göre azalan sırada sırala
2. İlk iki işi en iyi sırada yerleştir
3. Her yeni işi mevcut dizinin en iyi pozisyonuna ekle

In [ ]:
def neh(processing_times):
    """
    NEH sezgiseli ile Flow-Shop başlangıç çözümü üretir.
    
    Returns:
        permutation (list): İş sırası
        cmax (int): Bu sıranın makespan değeri
    """
    n_machines = len(processing_times)
    n_jobs = len(processing_times[0])

    total_times = []
    for j in range(n_jobs):
        total = sum(processing_times[i][j] for i in range(n_machines))
        total_times.append((total, j))
    total_times.sort(reverse=True)
    sorted_jobs = [job for _, job in total_times]

    permutation = [sorted_jobs[0]]
    for idx in range(1, n_jobs):
        job = sorted_jobs[idx]
        best_cmax = float('inf')
        best_pos = 0
        for pos in range(len(permutation) + 1):
            candidate = permutation[:pos] + [job] + permutation[pos:]
            cmax = calculate_makespan(candidate, processing_times)
            if cmax < best_cmax:
                best_cmax = cmax
                best_pos = pos
        permutation = permutation[:best_pos] + [job] + permutation[best_pos:]

    return permutation, calculate_makespan(permutation, processing_times)


# ── Test: tai20x5 ─────────────────────────────────────────────────────────────
pt = INSTANCES['tai20x5']
perm, cmax = neh(pt)
bks = BKS['tai20x5']
print(f"tai20x5 — NEH Cmax: {cmax}  |  BKS: {bks}  |  Gap: {(cmax-bks)/bks*100:+.2f}%")

---
## 5. Komşuluk Yapısı

**Swap hamlesi:** permütasyondaki iki işin pozisyonu yer değiştirilir.

- n ≤ 30: Tam komşuluk taranır
- n > 30: 80 rastgele komşu örneklenir (hesaplama süresi kontrolü)

In [ ]:
import random

FULL_SEARCH_THRESHOLD = 30
MAX_NEIGHBOURS = 80


def apply_swap(permutation, i, j):
    p = permutation[:]
    p[i], p[j] = p[j], p[i]
    return p


def get_best_neighbour(permutation, processing_times, tabu_list, best_known_cmax):
    """
    Tabu listesini dikkate alarak en iyi komşuyu bulur.
    Aspirasyon kriteri: BKS'yi geçen tabu hamle kabul edilir.
    """
    n = len(permutation)
    best_cmax = float('inf')
    best_neighbour = None
    best_move = None

    all_pairs = [(i, j) for i in range(n - 1) for j in range(i + 1, n)]
    pairs = random.sample(all_pairs, min(MAX_NEIGHBOURS, len(all_pairs))) \
            if n > FULL_SEARCH_THRESHOLD else all_pairs

    for i, j in pairs:
        move = (permutation[i], permutation[j], 'swap')
        neighbour = apply_swap(permutation, i, j)
        cmax = calculate_makespan(neighbour, processing_times)
        is_tabu = move in tabu_list
        aspiration = cmax < best_known_cmax
        if (not is_tabu or aspiration) and cmax < best_cmax:
            best_cmax = cmax
            best_neighbour = neighbour
            best_move = move

    return best_neighbour, best_cmax, best_move


print("Komşuluk yapısı tanımlandı.")

---
## 6. Tabu Search Algoritması

| Bileşen | Açıklama |
|---|---|
| Başlangıç | NEH sezgiseli |
| Komşuluk | Swap (örneklemeli) |
| Tabu Listesi | Son k hamleyi yasaklar |
| Aspirasyon | BKS'yi geçen tabu hamle kabul edilir |
| Durdurma | Max iterasyon veya iyileşme yok |

In [ ]:
from collections import deque


def tabu_search(processing_times, tenure=50, max_iter=600,
                no_improve_limit=120, verbose=False):
    """
    Tabu Search ile Flow-Shop Scheduling çözer.
    
    Args:
        processing_times: İşlem süresi matrisi
        tenure: Tabu listesi uzunluğu
        max_iter: Maksimum iterasyon
        no_improve_limit: Bu kadar iterasyon iyileşme olmazsa dur
    Returns:
        best_perm, best_cmax, history
    """
    current_perm, current_cmax = neh(processing_times)
    best_perm = current_perm[:]
    best_cmax = current_cmax

    tabu_list = deque(maxlen=tenure)
    history = [current_cmax]
    no_improve = 0

    for iteration in range(1, max_iter + 1):
        neighbour, neighbour_cmax, move = get_best_neighbour(
            current_perm, processing_times, tabu_list, best_cmax)

        if neighbour is None:
            break

        current_perm = neighbour
        current_cmax = neighbour_cmax
        if move:
            tabu_list.append(move)

        if current_cmax < best_cmax:
            best_cmax = current_cmax
            best_perm = current_perm[:]
            no_improve = 0
        else:
            no_improve += 1

        history.append(current_cmax)

        if verbose and iteration % 100 == 0:
            print(f"  İter {iteration:4d} | Mevcut: {current_cmax} | En İyi: {best_cmax}")

        if no_improve >= no_improve_limit:
            break

    return best_perm, best_cmax, history


# ── Hızlı test ────────────────────────────────────────────────────────────────
import time
random.seed(42)
pt = INSTANCES['tai20x5']
start = time.time()
_, cmax, history = tabu_search(pt, tenure=50, max_iter=600, verbose=True)
elapsed = time.time() - start
print(f"\ntai20x5 → Cmax={cmax}  BKS={BKS['tai20x5']}  Gap={((cmax-BKS['tai20x5'])/BKS['tai20x5']*100):+.2f}%  Süre={elapsed:.2f}s")

---
## 7. Parametre Tuning

tai50x10 instance'ı üzerinde **5 farklı tenure** değeri test edilmiştir (max_iter=600 sabit).

Her kombinasyon için **5 bağımsız çalıştırma** yapılmıştır.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Tuning sonuçları (önceden çalıştırılmış)
tenures  = [5,    10,   20,   30,   50  ]
gaps     = [9.62, 9.36, 9.40, 9.47, 9.32]
stds     = [17.2, 27.9, 18.4, 18.1,  7.3]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(tenures, gaps, 'o-', color='#00A896', linewidth=2, markersize=8)
ax1.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Seçilen: 50')
ax1.set_title('Tenure vs Optimality Gap\n(tai50x10, max_iter=600)', fontweight='bold')
ax1.set_xlabel('Tabu Tenure'); ax1.set_ylabel('Gap (%)')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_facecolor('#F8FAFB')

ax2.bar(tenures, stds, color='#0D9488', alpha=0.75, width=5)
ax2.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='Seçilen: 50')
ax2.set_title('Tenure vs Standart Sapma\n(Tutarlılık Analizi)', fontweight='bold')
ax2.set_xlabel('Tabu Tenure'); ax2.set_ylabel('Std Dev')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y'); ax2.set_facecolor('#F8FAFB')

plt.suptitle('Parametre Tuning Sonuçları', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/tuning_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Seçilen parametreler: tenure=50, max_iter=600")
print("Gerekçe: En düşük gap (%9.32) + En düşük std (7.3)")

---
## 8. Resmi Deneyler

**Seçilen parametreler:** tenure=50, max_iter=600, no_improve_limit=120

Her instance için **10 bağımsız çalıştırma** yapılmaktadır.

In [ ]:
import statistics
import csv

random.seed(42)
PARAMS = {'tenure': 50, 'max_iter': 600, 'no_improve_limit': 120}
N_RUNS = 10
DIFFICULTY = {'tai20x5':'Kolay','tai50x10':'Orta','tai100x10':'Zor','tai100x20':'Zor','tai200x20':'Çok Zor'}

os.makedirs('results', exist_ok=True)
os.makedirs('plots', exist_ok=True)

all_records = []
all_histories = {}
summary = {}

print('ENM404 Grup 9 | Flow-Shop + Tabu Search | Resmi Deneyler')
print('=' * 60)

for name, info in TAILLARD_CATALOG.items():
    bks = info['bks']
    pt  = INSTANCES[name]
    print(f'\n[{DIFFICULTY[name]}] {name}  ({info["n_jobs"]}x{info["n_machines"]})  BKS={bks}')

    cmaxs, times, hists = [], [], []
    for run in range(1, N_RUNS + 1):
        start = time.time()
        _, cmax, hist = tabu_search(pt, **PARAMS)
        elapsed = round(time.time() - start, 2)
        cmaxs.append(cmax); times.append(elapsed); hists.append(hist)
        print(f'  Çalıştırma {run:2d}: Cmax={cmax}  t={elapsed}s')
        all_records.append({'instance': name, 'run': run, 'cmax': cmax, 'time': elapsed})

    best  = min(cmaxs); worst = max(cmaxs)
    mean  = statistics.mean(cmaxs)
    std   = statistics.stdev(cmaxs) if len(cmaxs) > 1 else 0.0
    gap   = (best - bks) / bks * 100
    avg_t = statistics.mean(times)

    summary[name] = {'bks':bks,'best':best,'worst':worst,
                     'mean':round(mean,1),'std':round(std,1),
                     'gap':round(gap,2),'avg_t':round(avg_t,2)}
    all_histories[name] = hists
    print(f'  → Best={best}  Mean={mean:.1f}  Std={std:.1f}  Gap={gap:+.2f}%  AvgT={avg_t:.2f}s')

# CSV kaydet
with open('results/all_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['instance','run','cmax','time'])
    writer.writeheader(); writer.writerows(all_records)
print('\nCSV kaydedildi: results/all_results.csv')

In [ ]:
# ── Özet Tablo ────────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"{'Instance':<12} {'Zorluk':<10} {'BKS':>6} {'Best':>6} {'Mean':>7} {'Std':>6} {'Gap%':>7} {'T(s)':>7}")
print(f"{'='*75}")
for name, s in summary.items():
    print(f"{name:<12} {DIFFICULTY[name]:<10} {s['bks']:>6} {s['best']:>6} {s['mean']:>7} {s['std']:>6} {s['gap']:>+7.2f}% {s['avg_t']:>7}")
print(f"{'='*75}")

In [ ]:
# ── Yakınsama Grafikleri ──────────────────────────────────────────────────────
colors = ['#00A896','#0D9488','#F59E0B','#EF4444','#8B5CF6']
names  = list(TAILLARD_CATALOG.keys())

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, name, color in zip(axes, names, colors):
    bks  = BKS[name]
    hist = all_histories[name][0]
    gap_hist = [(v - bks) / bks * 100 for v in hist]
    ax.plot(gap_hist, color=color, linewidth=1.8)
    ax.axhline(y=0, color='red', linewidth=1, linestyle='--', alpha=0.6, label='BKS')
    ax.set_title(f'{name}\n({TAILLARD_CATALOG[name]["n_jobs"]}x{TAILLARD_CATALOG[name]["n_machines"]})',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('İterasyon', fontsize=9)
    if ax == axes[0]: ax.set_ylabel('Gap (%)', fontsize=9)
    ax.grid(True, alpha=0.3); ax.set_facecolor('#F8FAFB')
    ax.legend(fontsize=8)

fig.suptitle('Tabu Search — Yakınsama Grafikleri (Optimality Gap %)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/convergence_all.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Karşılaştırmalı Analiz

Tabu Search, iki baseline yöntemiyle karşılaştırılmaktadır:
1. **Random Search** — rastgele permütasyon dener
2. **NEH Only** — sadece başlangıç sezgiseli, iyileştirme yok

In [ ]:
import numpy as np

def random_search(pt, n_jobs, max_iter=600):
    jobs = list(range(n_jobs))
    best = float('inf')
    for _ in range(max_iter):
        perm = jobs[:]; random.shuffle(perm)
        cmax = calculate_makespan(perm, pt)
        if cmax < best: best = cmax
    return best

random.seed(42)
print(f"{'Instance':<12} {'BKS':>6} {'Random':>8} {'NEH':>8} {'TS Best':>8} {'R_gap%':>9} {'N_gap%':>9} {'TS_gap%':>9}")
print('='*75)

random_gaps, neh_gaps, ts_gaps_list = [], [], []

for name, info in TAILLARD_CATALOG.items():
    bks = info['bks']; pt = INSTANCES[name]
    rs_best  = random_search(pt, info['n_jobs'], max_iter=300)
    _, neh_c = neh(pt)
    ts_best  = summary[name]['best']
    r_gap  = (rs_best  - bks) / bks * 100
    n_gap  = (neh_c   - bks) / bks * 100
    ts_gap = (ts_best  - bks) / bks * 100
    random_gaps.append(r_gap); neh_gaps.append(n_gap); ts_gaps_list.append(ts_gap)
    print(f"{name:<12} {bks:>6} {rs_best:>8} {neh_c:>8} {ts_best:>8} {r_gap:>+9.2f}% {n_gap:>+9.2f}% {ts_gap:>+9.2f}%")

print('='*75)
print(f"\nTS ortalama iyileştirmesi (NEH'e kıyasla): {sum(n-t for n,t in zip(neh_gaps,ts_gaps_list))/len(neh_gaps):+.2f} pp")

In [ ]:
# ── Karşılaştırma Grafiği ─────────────────────────────────────────────────────
x = np.arange(len(names)); w = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

b1 = ax1.bar(x - w, random_gaps, w, label='Random Search', color='#E74C3C', alpha=0.8)
b2 = ax1.bar(x,     neh_gaps,    w, label='NEH Only',      color='#F39C12', alpha=0.8)
b3 = ax1.bar(x + w, ts_gaps_list,w, label='Tabu Search',   color='#00A896', alpha=0.8)
ax1.axhline(y=0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=15, fontsize=9)
ax1.set_ylabel('Optimality Gap (%)'); ax1.legend()
ax1.set_title('Yöntem Karşılaştırması — Optimality Gap', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y'); ax1.set_facecolor('#F8FAFB')

improvements = [n - t for n, t in zip(neh_gaps, ts_gaps_list)]
colors_bar = ['#2ecc71' if v > 0 else '#e74c3c' for v in improvements]
ax2.bar(names, improvements, color=colors_bar, alpha=0.8, edgecolor='white')
for i, (name, val) in enumerate(zip(names, improvements)):
    ax2.text(i, val + 0.05, f'{val:+.2f}pp', ha='center', fontsize=9, fontweight='bold')
ax2.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax2.set_title("Tabu Search'in NEH'e Göre İyileştirmesi", fontweight='bold')
ax2.set_ylabel('İyileştirme (pp)'); ax2.set_xticklabels(names, rotation=15, fontsize=9)
ax2.grid(True, alpha=0.3, axis='y'); ax2.set_facecolor('#F8FAFB')

plt.tight_layout()
plt.savefig('plots/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Sonuçlar

### Özet

| Bulgu | Değer |
|---|---|
| Toplam çalıştırma | 48 run |
| BKS altına inen instance | 2 / 5 |
| Ortalama optimality gap | +1.30% |
| NEH'e ortalama iyileştirme | +1.83 pp |
| Seçilen parametreler | tenure=50, max_iter=600 |

### Güçlü Yönler
- NEH başlangıcı hızlı yakınsama sağlar
- Küçük/zor instance'larda BKS altına inilebiliyor
- Tutarlı sonuçlar (düşük std)
- Random Search'e her instance'da belirgin üstünlük

### Zayıf Yönler
- tai50x10'da yüksek gap (+9.36%)
- Büyük instance'larda örnekleme kaliteyi kısıtlıyor
- 200x20'de ~23s/run pratik sınıra yakın

### Kaynaklar
- Taillard, E. (1993). Benchmarks for basic scheduling problems. *EJOR*, 64(2), 278–285.
- Nawaz, M., Enscore, E. E., & Ham, I. (1983). A heuristic algorithm for the m-machine, n-job flow-shop. *Omega*, 11(1), 91–95.
- Glover, F. (1989). Tabu Search — Part I. *ORSA Journal on Computing*, 1(3), 190–206.